In [1]:
#froot = r"D:\pAce\BKV009\20260512\FOV1_T14"
froot = r"D:\pAce\BKV009\20260703\FOV1_T4"
#froot = r"D:\pAce\BKV008\20260703\FOV1_T3"
analysis_mode = "new"  # or "cellpose" for cell segmentation
folder_paths = [froot]

In [2]:

print("Importing packages and Initializing...")
version="V1"

from pathlib import Path
current_dir = r"C:\Users\strenglab\VoImAn\caiman\ICNLAB"
weights_path= str(current_dir + "//" + "mask_rcnn_neuron_0012.h5")

#V1.2: 0.8 corr cutoff, 2 minimum ratio of h over w for spikes, cell_idxs incremented by 1, wheel data appended to mat save
print("version:", version)

import matplotlib
matplotlib.use("QtAgg")   # interactive, no windows
print(matplotlib.get_backend())

from base64 import b64encode
import cv2
import glob
import h5py
import imageio
from IPython import get_ipython
from IPython.display import HTML, display, clear_output
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from PIL import Image
import re
import csv
from datetime import datetime

#import to cover extras from single_trial.py
import gc
import scipy.io
from scipy import stats
from scipy.signal import butter, lfilter
from scipy.signal import savgol_filter
import sys
import mat73
import pandas as pd


from pathlib import Path

try:
    cv2.setNumThreads(0)
except:
    pass

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
        get_ipython().run_line_magic('matplotlib', 'qt')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.utils.utils import download_demo, download_model
from caiman.source_extraction.volpy import utils
from caiman.source_extraction.volpy.volparams import volparams
from caiman.source_extraction.volpy.volpy import VOLPY
from caiman.source_extraction.volpy.mrcnn import visualize, neurons
import caiman.source_extraction.volpy.mrcnn.model as modellib
from caiman.summary_images import local_correlations_movie_offline
from caiman.summary_images import mean_image
from caiman.paths import caiman_datadir
from caiman.summary_images import local_correlations_movie_in_memory
import gc
from caiman.ICNLAB.single_trial_simple_plotting_streng import plotdata

logging.basicConfig(format=
                    "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s]" \
                    "[%(process)d] %(message)s",
                    level=logging.ERROR)


Importing packages and Initializing...
version: V1
QtAgg


In [3]:

##BEGIN MAIN ANALYSIS LOOP
folder_path = folder_paths[0]  # Select the first folder path

#print to log even if no tsm
rootpath = Path(folder_path).parts[0] + Path(folder_path).parts[-4] + '\\Analysis\\'
Path(rootpath).mkdir(parents=True, exist_ok=True)
log_csv_path = Path(rootpath) / "MasterAnalysisLOG.csv" #Master CSV path
unique_save_string1 = "-".join(Path(folder_path).parts[-3:])

# find the .tsm file in the folder
tsm_files = [f for f in os.listdir(folder_path) if f.endswith(('.tsm', '.dcimg'))]
if not tsm_files:
    print(f"No recording files found in {folder_path}, skipping.")
    #Append new row to MASTERLOG
    today_str = datetime.now().strftime("%Y%m%d%H%M%S")  # compact datetime string
    new_row = [version, today_str, unique_save_string1, "No recording file"]
    with open(log_csv_path, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(new_row)
    print(f"Added new ERROR row to MasterAnalysisLOG.csv: {new_row}")
#continue if more than one .tsm file found
if len(tsm_files) > 1:
    print(f"Multiple recording files found in {folder_path}, skipping.")
    #Append new row to MASTERLOG
    today_str = datetime.now().strftime("%Y%m%d%H%M%S")  # compact datetime string
    new_row = [version, today_str, unique_save_string1, "Multiple recording files"]
    with open(log_csv_path, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(new_row)
    print(f"Added new ERROR row to MasterAnalysisLOG.csv: {new_row}")



fname = os.path.join(folder_path, tsm_files[0])
print('fname is', fname)
print("Processing file:", fname)



fname is D:\pAce\BKV009\20260703\FOV1_T4\FOV1_T4_Green.dcimg
Processing file: D:\pAce\BKV009\20260703\FOV1_T4\FOV1_T4_Green.dcimg


In [4]:


fpath = Path(fname)
#Create new unique save name
unique_save_string = "-".join(fpath.parts[-4:-1])
rootpath = str(Path(*fpath.parts[:-4]))+'\\Analysis\\'
print("Unique save string:", unique_save_string)
print("Directory for Analysis Files:", rootpath)
Path(rootpath).mkdir(parents=True, exist_ok=True)
log_csv_path = Path(rootpath) / "MasterAnalysisLOG.csv" #Master CSV path

#grab data for plotting with single_trial_simple_plotting.py
mouseID = fpath.parts[-4]
date = fpath.parts[-3]
trialname = fpath.parts[-2]

##
#fname = r'C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM2\FOV1_T2.tsm'
fr = 3600  ################################################################REMOVE LATER\
H, W = 1108,18
print(fname, fr)


##
# Cleanup R:/ drive (temp RAM disk)
print("Cleaning up R:/ drive...")
def safe_close_mmap(arr):
    try:
        if hasattr(arr, "base") and hasattr(arr.base, "close"):
            arr.base.close()
    except Exception as e:
        print("close failed:", e)


# 1. Delete any Python references to memmaps pointing to R:/
try:
    safe_close_mmap(Yr)  # or whatever your memmap object is called
except NameError:
    pass

try:
    safe_close_mmap(mmap_file_rig)  # or whatever your memmap object is called
except NameError:
    pass

gc.collect()  # force Python to release the memory mapping

# 2. Delete all files in R:/
for f in Path(r'R:/').glob('*'):
    if f.is_file():
        f.unlink()
print("Cleared all files from R:/")


##
pw_rigid = False  # flag for pw-rigid motion correction
gsig_filt = (3, 3)  # size of filter, in general gSig (see below),
# change this one if algorithm does not work
max_shifts = (5, 5)  # maximum allowed rigid shift
strides = (48, 48)  # start a new patch for pw-rigid motion correction every x pixels
overlaps = (24, 24)  # overlap between paths (size of patch strides+overlaps)
max_deviation_rigid = 3  # maximum deviation allowed for patch with respect to rigid shifts
border_nan = 'copy'
use_cuda = True

opts_dict = {
    'fnames': fname,
    'fr': fr,
    'pw_rigid': pw_rigid,
    'max_shifts': max_shifts,
    'gSig_filt': gsig_filt,
    'strides': strides,
    'overlaps': overlaps,
    'max_deviation_rigid': max_deviation_rigid,
    'border_nan': border_nan,
    'use_cuda': use_cuda
}

opts = volparams(params_dict=opts_dict)

##
print("Loading data...")
m_orig = cm.load(fname)
ds_ratio = 0.2

##
try:
    c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False)
except:
    print("Cluster running doing restart")
    dview.terminate()
    c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False)
    
##
print("Motion correction...")
mc = MotionCorrect(fname, dview=dview, **opts.get_group('motion'))
mc.motion_correct(save_movie=True, save_dir="R:/")
#about 2.3 minutes for 12800 frames (2m 13-21 s)
print("Done.")

##
print("Loading corrected movie...")
m_rig = cm.load(mc.mmap_file) # 11s
ds_ratio = 0.2
print("Done.")

del m_orig
gc.collect()

####CONVERT VIA STREAMING WITH HOPEFULLY SAME OG BEHAVIOR
p = Path(fname)

ram_path = Path(r'R:/') / (
    f"{p.stem}_rig__d1_{m_rig.shape[1]}"
    f"_d2_{m_rig.shape[2]}"
    f"_d3_1_order_C_frames_{m_rig.shape[0]}.mmap"
)
ram_path = str(ram_path).replace("/", "\\")

# Destination memmap: SAME AS ORIGINAL
dst = np.memmap(
    ram_path,
    dtype='float32',
    mode='w+',
    shape=m_rig.shape,
    order='F'   # critical: this is what caused the layout change originally
)

# Streaming copy (logical copy, not byte copy)
chunk = 16  # frames per chunk; tune for cache / IO

T = m_rig.shape[0]

for t0 in range(0, T, chunk):
    t1 = min(t0 + chunk, T)
    dst[t0:t1] = m_rig[t0:t1]

dst.flush()
mmap_list = [dst]

if hasattr(dst, 'base') and hasattr(dst.base, 'close'):
    dst.base.close()
del dst
gc.collect()

##


Unique save string: BKV009-20260703-FOV1_T4
Directory for Analysis Files: D:\pAce\Analysis\
D:\pAce\BKV009\20260703\FOV1_T4\FOV1_T4_Green.dcimg 3600
Cleaning up R:/ drive...
Cleared all files from R:/
Loading data...
Motion correction...
Saving mmap to:  R:/FOV1_T4_Green_rig__d1_18_d2_1108_d3_1_order_F_frames_108000.mmap
Done.
Loading corrected movie...


100%|██████████| 1/1 [00:05<00:00,  5.57s/it]


Done.


0

In [5]:
print(ram_path)

R:\FOV1_T4_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_108000.mmap


In [6]:

print("Computing mean and correlation images...")
img = np.mean(m_rig, axis=0)
img = (img-np.mean(img))/np.std(img)

#Display img
plt.figure(figsize=(15, 2))
plt.imshow(img, cmap="viridis")
plt.colorbar()
plt.show()
print(img.shape)

Computing mean and correlation images...
(18, 1108)


In [7]:

# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.signal import butter, filtfilt
# from tqdm import tqdm

# # ===============================
# # 1. Parameters
# # ===============================
# HIGHPASS_THRESH = (5)
# shape = m_rig.shape
# # ===============================
# # 2. Load memory-mapped video
# # ===============================

# video = np.memmap(
#     mc.mmap_file[0],
#     dtype=np.float32,
#     mode="r",
#     shape=shape,
#     order="C"
# ).swapaxes(1, 2)

# T, H, W = video.shape
# print(f"Loaded video: {video.shape}")

# # ===============================
# # 3. High-pass filter (bandpass-compatible API)
# # ===============================
# def highpass_filter(data, fs, low, high=None, order=3):
#     """
#     High-pass filter using the 'low' cutoff.
#     The 'high' argument is accepted for API compatibility but ignored.
#     """
#     nyq = 0.5 * fs
#     b, a = butter(order, low / nyq, btype="high")
#     return filtfilt(b, a, data, axis=0)


# # ===============================
# # Parameters
# # ===============================
# TILE_SIZE = 2
# H, W = H, W
# FRAME_RATE = fr
# # (low, high), high ignored
# DISPLAY_CLIP = 99

# # ===============================
# # Coherence metric
# # ===============================
# def coherence_metric(tile_filt):
#     """
#     tile_filt: shape (T, Npix)
#     Returns mean pixel-to-tile correlation.
#     """
#     # Tile reference (subthreshold signals sum coherently)
#     ref = tile_filt.mean(axis=1)


#     ref -= ref.mean()
#     ref_std = ref.std() + 1e-9

#     # Normalize reference
#     ref /= ref_std

#     # Normalize pixels
#     pix = tile_filt - tile_filt.mean(axis=0)
#     pix /= (pix.std(axis=0) + 1e-9)

#     # Correlation with reference
#     corr = np.mean(ref[:, None] * pix, axis=0)

#     # Use mean absolute correlation as coherence
#     return np.mean(np.abs(corr))


# # ===============================
# # Output tile map
# # ===============================
# n_tiles_y = H // TILE_SIZE
# n_tiles_x = W // TILE_SIZE

# tile_coherence_map = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)

# # ===============================
# # Main loop
# # ===============================
# with tqdm(total=n_tiles_y * n_tiles_x, desc="Computing coherence") as pbar:
#     for ty in range(n_tiles_y):
#         for tx in range(n_tiles_x):

#             y0 = ty * TILE_SIZE
#             y1 = y0 + TILE_SIZE
#             x0 = tx * TILE_SIZE
#             x1 = x0 + TILE_SIZE

#             # Extract tile: (T, 16, 16)
#             tile = video[:, y0:y1, x0:x1]
#             tile = tile.reshape(T, -1)

#             # High-pass filter all pixels independently
#             tile_filt = highpass_filter(
#                 tile, FRAME_RATE, HIGHPASS_THRESH
#             )

#             # Compute coherence
#             tile_coherence_map[ty, tx] = coherence_metric(tile_filt)

#             pbar.update(1)

# # ===============================
# # Expand to image resolution
# # ===============================
# coherence_image = np.repeat(
#     np.repeat(tile_coherence_map, TILE_SIZE, axis=0),
#     TILE_SIZE, axis=1
# )

# if hasattr(video, 'base') and hasattr(video.base, 'close'):
#     video.base.close()

# del video
# gc.collect()


In [8]:
#NEW STRENG LAB VERSION CHANGED LOADING TO ORDER F AND SOS HIGH PASS FILTERING
#DEBUG LOOP WITH 3 TEST CUT OFFS
#  import numpy as np
# import matplotlib.pyplot as plt
# from scipy.signal import butter, sosfiltfilt
# from tqdm import tqdm
# import gc

# # ===============================
# # 1. Configuration & Parameters
# # ===============================
# # We assume m_rig.shape is (T, H, W), e.g., (36000, 18, 1108)
# T_frames, H_dim, W_dim = m_rig.shape 
# FRAME_RATE = 3600
# TILE_SIZE = 2
# TEST_CUTOFFS = [2.0, 5.0, 10.0]

# # ===============================
# # 2. Load Memory-Mapped Video
# # ===============================
# print("Loading memory-mapped file...")

# # Load the file in the exact way CaImAn wrote it to disk: (H, W, T) and order="F"
# video_raw = np.memmap(
#     mc.mmap_file[0],
#     dtype=np.float32,
#     mode="r",
#     shape=(H_dim, W_dim, T_frames),
#     order="F"
# )

# # Transpose the axes to get it back to Python's preferred (Time, Height, Width)
# video = np.transpose(video_raw, (2, 0, 1))

# print(f"Correctly oriented video: {video.shape}")

# # ===============================
# # 3. Core Logic Function
# # ===============================
# def compute_coherence_map(vid_data, fs, cutoff, tile_size):
#     t_frames, height, width = vid_data.shape
#     n_tiles_y = height // tile_size
#     n_tiles_x = width // tile_size
    
#     tile_coherence_map = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)
    
#     # Use SOS (Second-Order Sections) for numerical stability at high sampling rates
#     nyq = 0.5 * fs
#     sos = butter(3, cutoff / nyq, btype="high", output="sos")
    
#     with tqdm(total=n_tiles_y * n_tiles_x, desc=f"Processing {cutoff}Hz cutoff") as pbar:
#         for ty in range(n_tiles_y):
#             for tx in range(n_tiles_x):
#                 y0, y1 = ty * tile_size, (ty + 1) * tile_size
#                 x0, x1 = tx * tile_size, (tx + 1) * tile_size

#                 # Extract and flatten spatial dimensions: (T, Npix)
#                 tile = vid_data[:, y0:y1, x0:x1].reshape(t_frames, -1)

#                 # High-pass filter using stable SOS
#                 tile_filt = sosfiltfilt(sos, tile, axis=0)

#                 # Coherence metric calculation
#                 ref = tile_filt.mean(axis=1)
#                 ref -= ref.mean()
#                 ref /= (ref.std() + 1e-9)

#                 pix = tile_filt - tile_filt.mean(axis=0)
#                 pix /= (pix.std(axis=0) + 1e-9)

#                 corr = np.mean(ref[:, None] * pix, axis=0)
#                 tile_coherence_map[ty, tx] = np.mean(np.abs(corr))

#                 pbar.update(1)
                
#     # Expand to image resolution
#     expanded_map = np.repeat(
#         np.repeat(tile_coherence_map, tile_size, axis=0),
#         tile_size, axis=1
#     )
#     return expanded_map

# # ===============================
# # 4. Main Loop & Plotting
# # ===============================
# results = []
# for cutoff in TEST_CUTOFFS:
#     cmap = compute_coherence_map(video, FRAME_RATE, cutoff, TILE_SIZE)
#     results.append((cutoff, cmap))

# # Clean up memory map before plotting to free RAM
# print("Cleaning up memory...")
# if hasattr(video_raw, 'base') and hasattr(video_raw.base, 'close'):
#     video_raw.base.close()
# del video
# del video_raw
# gc.collect()

# # Plotting the results side-by-side
# print("Generating plots...")
# fig, axes = plt.subplots(1, len(TEST_CUTOFFS), figsize=(18, 8))

# # Ensure axes is iterable even if there's only 1 cutoff test
# if len(TEST_CUTOFFS) == 1:
#     axes = [axes]

# for ax, (cutoff, cmap_data) in zip(axes, results):
#     # Transpose the data here so it is 1108 tall and 18 wide
#     plot_data = cmap_data.T 
    
#     # Use robust scaling to prevent outliers from washing out the image
#     vmin = np.nanpercentile(plot_data, 2)
#     vmax = np.nanpercentile(plot_data, 98)
    
#     # Using aspect="equal" to keep true pixel proportions without stretching the 18-pixel width
#     im = ax.imshow(plot_data, cmap="viridis", aspect="equal", vmin=vmin, vmax=vmax)
#     ax.set_title(f"High-pass Cutoff: {cutoff} Hz")
    
#     # Add colorbar 
#     fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# plt.tight_layout()
# plt.show()

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
from tqdm import tqdm
import gc

# ===============================
# 1. Configuration & Parameters
# ===============================
# We assume m_rig.shape is (T, H, W), e.g., (36000, 18, 1108)
T_frames, H_dim, W_dim = m_rig.shape 
FRAME_RATE = fr
TILE_SIZE = 2
CUTOFF = 5.0  # Run only for 5Hz cutoff

# ===============================
# 2. Load Memory-Mapped Video
# ===============================
print("Loading memory-mapped file...")

# Load the file in the exact way CaImAn wrote it to disk: (H, W, T) and order="F"
video_raw = np.memmap(
    mc.mmap_file[0],
    dtype=np.float32,
    mode="r",
    shape=(H_dim, W_dim, T_frames),
    order="F"
)

# Transpose the axes to get it back to Python's preferred (Time, Height, Width)
video = np.transpose(video_raw, (2, 0, 1))

print(f"Correctly oriented video: {video.shape}")

# ===============================
# 3. Core Logic Function
# ===============================
def compute_coherence_map(vid_data, fs, cutoff, tile_size):
    t_frames, height, width = vid_data.shape
    n_tiles_y = height // tile_size
    n_tiles_x = width // tile_size
    
    tile_coherence_map = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)
    
    # Use SOS (Second-Order Sections) for numerical stability at high sampling rates
    nyq = 0.5 * fs
    sos = butter(3, cutoff / nyq, btype="high", output="sos")
    
    with tqdm(total=n_tiles_y * n_tiles_x, desc=f"Processing {cutoff}Hz cutoff") as pbar:
        for ty in range(n_tiles_y):
            for tx in range(n_tiles_x):
                y0, y1 = ty * tile_size, (ty + 1) * tile_size
                x0, x1 = tx * tile_size, (tx + 1) * tile_size

                # Extract and flatten spatial dimensions: (T, Npix)
                tile = vid_data[:, y0:y1, x0:x1].reshape(t_frames, -1)

                # High-pass filter using stable SOS
                tile_filt = sosfiltfilt(sos, tile, axis=0)

                # Coherence metric calculation
                ref = tile_filt.mean(axis=1)
                ref -= ref.mean()
                ref /= (ref.std() + 1e-9)

                pix = tile_filt - tile_filt.mean(axis=0)
                pix /= (pix.std(axis=0) + 1e-9)

                corr = np.mean(ref[:, None] * pix, axis=0)
                tile_coherence_map[ty, tx] = np.mean(np.abs(corr))

                pbar.update(1)
                
    # Expand to image resolution
    expanded_map = np.repeat(
        np.repeat(tile_coherence_map, tile_size, axis=0),
        tile_size, axis=1
    )
    return expanded_map

# ===============================
# 4. Main Run & Plotting
# ===============================
cmap_data = compute_coherence_map(video, FRAME_RATE, CUTOFF, TILE_SIZE)

# # Clean up memory map before plotting to free RAM
# print("Cleaning up memory...")
# if hasattr(video_raw, 'base') and hasattr(video_raw.base, 'close'):
#     video_raw.base.close()
# del video
# del video_raw
# gc.collect()

# Plotting the result


Loading memory-mapped file...
Correctly oriented video: (108000, 18, 1108)


Processing 5.0Hz cutoff: 100%|██████████| 4986/4986 [01:19<00:00, 62.44it/s]


In [10]:
# print("Generating plot...")
# plt.figure(figsize=(10, 8))

# # Transpose the data here so it is 1108 tall and 18 wide
# plot_data = cmap_data.T 

# # Use robust scaling to prevent outliers from washing out the image
# vmin = np.nanpercentile(cmap_data, 2)
# vmax = np.nanpercentile(cmap_data, 98)

# # Using aspect="equal" to keep true pixel proportions without stretching the 18-pixel width
# im = plt.imshow(cmap_data, cmap="viridis", aspect="equal", vmin=vmin, vmax=vmax)
# plt.title(f"High-pass Cutoff: {CUTOFF} Hz")

# # Add colorbar 
# plt.colorbar(im, fraction=0.046, pad=0.04)

# plt.tight_layout()
# plt.show()

In [11]:
# import matplotlib.pyplot as plt
# import matplotlib.colors as colors
# import numpy as np

# # ---------------------------------------------------------
# # 1. Print Image Size and Stats
# # ---------------------------------------------------------
# print(f"Coherence image shape: {coherence_image.shape}")
# print(f"Total pixels: {coherence_image.size:,}")
# # Using np.nanmin/nanmax just in case your data contains any NaN values
# print(f"Min intensity: {np.nanmin(coherence_image):.4f}")
# print(f"Max intensity: {np.nanmax(coherence_image):.4f}")
# print(f"Mean intensity: {np.nanmean(coherence_image):.4f}")
# print("-" * 30)

# # ---------------------------------------------------------
# # 2. Try Different Scalings side-by-side
# # ---------------------------------------------------------
# fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# # Calculate "robust" limits (ignoring the bottom 1% and top 1% of extreme pixels)
# robust_vmin = np.nanpercentile(coherence_image, 1)
# robust_vmax = np.nanpercentile(coherence_image, 99)

# # A. Robust Linear Scaling (Clips extreme outliers)
# im0 = axes[0].imshow(coherence_image, cmap="viridis", aspect="auto", 
#                      vmin=robust_vmin, vmax=robust_vmax)
# axes[0].set_title("Robust Linear (1st-99th Percentile)")
# fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# # B. Logarithmic Scaling (Great if data has a massive dynamic range)
# # Log scaling doesn't like zeros/negatives, so we add a tiny epsilon just in case
# epsilon = 1e-6 
# valid_log_min = max(robust_vmin, epsilon)
# im1 = axes[1].imshow(coherence_image + epsilon, cmap="viridis", aspect="auto", 
#                      norm=colors.LogNorm(vmin=valid_log_min, vmax=robust_vmax))
# axes[1].set_title("Logarithmic Scale")
# fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# # C. Power-Law / Gamma Scaling (gamma < 1 boosts dark pixels, > 1 boosts brights)
# im2 = axes[2].imshow(coherence_image, cmap="viridis", aspect="auto", 
#                      norm=colors.PowerNorm(gamma=0.5, vmin=np.nanmin(coherence_image), vmax=robust_vmax))
# axes[2].set_title("Power Norm (Gamma=0.5)")
# fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

# plt.tight_layout()
# plt.show()

# # ---------------------------------------------------------
# # 3. Plot the Histogram of Intensity Values
# # ---------------------------------------------------------
# plt.figure(figsize=(10, 5))

# # .ravel() flattens the 11008x18 array into a single 1D list for the histogram
# # We also filter out NaNs if any exist to prevent matplotlib errors
# valid_pixels = coherence_image[~np.isnan(coherence_image)] 

# # Plotting 100 bins. 
# plt.hist(valid_pixels.ravel(), bins=100, color='teal', edgecolor='black', alpha=0.7)

# plt.title("Histogram of Pixel Intensities")
# plt.xlabel("Intensity Value")
# plt.ylabel("Frequency (Number of Pixels)")

# # We set the Y-axis to log scale. Because you have ~200,000 pixels, 
# # a log scale prevents the tallest bar from squishing all the smaller bars into invisibility.
# plt.yscale('log') 
# plt.grid(axis='y', alpha=0.5)

# plt.show()

In [12]:

# # ===============================
# # Visualization
# # ===============================
# vmax = np.percentile(coherence_image, DISPLAY_CLIP)

# plt.figure(figsize=(6, 6))
# plt.imshow(coherence_image, cmap="viridis", vmin=0, vmax=vmax)
# plt.title("Grid-based subthreshold coherence ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
# plt.colorbar(label="Mean |pixel–tile correlation|")
# plt.axis("off")
# plt.tight_layout()
# plt.show()



In [13]:
# # Make the figure window significantly larger (width, height)
# plt.figure(figsize=(15, 10)) 

# # aspect="auto" stretches the 18 pixels so they are visible alongside the 11008
# plt.imshow(coherence_image, cmap="viridis", vmin=0, vmax=vmax, aspect="auto")

# plt.colorbar() # Highly recommend adding this if you are using vmin/vmax!
# plt.show()

In [14]:

# img_corr = cmap_data
# print(f"Coherence image shape: {cmap_data.shape}")
# print(f" image shape: {img.shape}")


In [15]:
# img_corr = cmap_data

# summary_images = np.stack([img, img, img_corr], axis=0).astype(np.float32)
# #cm.movie(summary_images).save(fname[:-5]+'_summary_images.tif')

# plt.imshow(summary_images[0], cmap='gray')
# plt.axis('off')
# #plt.savefig(fname[:-4]+'_mean.tif', format='tif', bbox_inches='tight', pad_inches=0)


# plt.imshow(summary_images[2], cmap='gray')
# plt.axis('off')
# #plt.savefig(fname[:-4]+'_corr.tif', format='tif', bbox_inches='tight', pad_inches=0)

# img = summary_images.transpose([1, 2, 0])


# print(fname[:-4]+'_corr.tif')
# height, width = img.shape[:2]
# print(img.shape)

# # --------------------------------------------------------------
# # Extract channels like MATLAB
# # --------------------------------------------------------------
# R = img[:, :, 0]
# B = img[:, :, 2]

# # --------------------------------------------------------------
# # MATLAB-style normalization (mat2gray + uint8)
# # --------------------------------------------------------------
# def normalize_like_matlab(x):
#     x = x.astype(np.float64)
#     mn = x.min()
#     mx = x.max()
#     x = (x - mn) / (mx - mn + 1e-12)

#     # MATLAB uint8 applies rounding, not floor
#     x = np.round(255 * x).astype(np.uint8)
#     return x

# R_norm = normalize_like_matlab(R)
# B_norm = normalize_like_matlab(B)

# # --------------------------------------------------------------
# # Build MATLAB-equivalent RGB (R,R,B)
# # --------------------------------------------------------------
# rgb = np.stack([R_norm, R_norm, B_norm], axis=2).astype(np.uint8)

# # --------------------------------------------------------------
# # Save as PNG/TIF (MATLAB-compatible pixel data)
# # --------------------------------------------------------------
# outname =  rootpath + unique_save_string + ".tif"
# Image.fromarray(rgb).save(outname)

# print("Saved:", outname)
# img = rgb.copy()



In [16]:
import skimage.io as skio

new_filename = "temp_processing_stack.tif"
# old_filename = os.path.basename(fname)
temp_tif_path = "R:/" + new_filename
print(f"Temporary TIF path: {temp_tif_path}")

# Output: /home/user/documents/new_file.txt
# ---------------------------------------------------------
# 2. Save Data to a TIF Stack
# ---------------------------------------------------------
print(f"Saving active imaging data stack to: {temp_tif_path}")
skio.imsave(temp_tif_path, video.astype('float32'), check_contrast=False)
print("TIF file successfully cached.")


Temporary TIF path: R:/temp_processing_stack.tif
Saving active imaging data stack to: R:/temp_processing_stack.tif
TIF file successfully cached.


In [17]:

import subprocess
import os
#
#RUN SUPPORT DENOISING IN A SEPARATE CONDA ENVIRONMENT
#


# ==============================================================================
# CONFIGURATION (Adjust anaconda_path if your Anaconda is installed elsewhere)
# ==============================================================================
# Common alternatives: r"C:\ProgramData\anaconda3\Scripts\activate.bat" or Miniconda equivalents
anaconda_activate_path = r"C:\Users\strenglab\anaconda3\Scripts\activate.bat" 
working_dir = r"C:\Users\strenglab\Documents\Python\SUPPORT"
conda_env = "SUPPORT"

# Inference Arguments
data_file = temp_tif_path  # This is the TIF we just saved to R:/
model_file = r"C:\Users\strenglab\Documents\Python\SUPPORT\results\saved_models\pAce_BKV009_20260512_FOV1_T14\model_10_batch_80000.pth"
output_file = temp_tif_path.replace(".tif", "_denoised.tif")

# ==============================================================================
# BUILD AND RUN COMMAND
# ==============================================================================

python_cmd = (
    f'python -m src.Denoise_callable ' # <-- Added -m and used dot notation (NO .py at the end!)
    f'-d "{data_file}" '
    f'-m "{model_file}" '
    f'-o "{output_file}" '
    f'-p 61 16 16 '
    f'-i 1 8 8 '
    f'-b 32'
)

# Notice the " || pause" added at the end
inner_command = (
    f'cd /d "{working_dir}" && '
    f'call "{anaconda_activate_path}" && '
    f'conda activate {conda_env} && '
    f'{python_cmd} || pause'
)

full_command = f'start "SUPPORT Denoising" /wait cmd.exe /c "{inner_command}"'

print("-" * 60)
print(f"Launching a new visible window for debugging...")
print(f"NOTE: Your script will pause here. You MUST manually close the new Command Prompt window for this script to continue.")
print("-" * 60)

subprocess.run(full_command, shell=True)

print("-" * 60)
print("New window was closed. Resuming the rest of the parent script execution...")
print("-" * 60)
print("--- Denoising complete! ---\n")
print("Resuming main pipeline processing loop...")

# Your downstream code (e.g., loading the output_path file) goes here:
# denoised_data = scipy.io.loadmat(output_path)

# Define paths
# import shutil


# source = r"D:\pAce\BKV009\20260512\FOV1_T14\TEMP\temp_processing_stack_denoised.tif"
# destination = "R:/temp_processing_stack_denoised.tif"

# # Copy file into the folder
# shutil.copy(source, destination)


------------------------------------------------------------
Launching a new visible window for debugging...
NOTE: Your script will pause here. You MUST manually close the new Command Prompt window for this script to continue.
------------------------------------------------------------
------------------------------------------------------------
New window was closed. Resuming the rest of the parent script execution...
------------------------------------------------------------
--- Denoising complete! ---

Resuming main pipeline processing loop...


In [18]:
# import numpy as np
# import matplotlib.pyplot as plt
# import itertools
# import gc

# # 1. Define the three known dimensions
# # Assuming m_rig is still in your environment. If not, replace these with your raw numbers.
# T = m_rig.shape[0]
# d1 = m_rig.shape[1]
# d2 = m_rig.shape[2]

# dimensions = [T, d1, d2]
# dim_names = ['T', 'd1', 'd2']

# # 2. Generate all 6 possible permutations
# perms = list(itertools.permutations(dimensions))
# name_perms = list(itertools.permutations(dim_names))

# # 3. Setup the plot
# fig, axes = plt.subplots(2, 3, figsize=(16, 10))
# axes = axes.flatten()
# fig.suptitle(f"Memmap Layout Diagnostic\nFile: {ram_path}", fontsize=16)

# # 4. Loop through all 6 permutations
# for i, (shape_perm, name_perm) in enumerate(zip(perms, name_perms)):
    
#     # Load the memmap with the current test shape in standard C-order
#     temp_mmap = np.memmap(
#         ram_path, 
#         dtype='float32', 
#         mode='r', 
#         shape=shape_perm, 
#         order='C'
#     )
    
#     # Find which axis currently represents Time ('T') so we project across the right dimension
#     time_axis = name_perm.index('T')
    
#     # Slice only the first 50 frames to make this run instantly
#     slices = [slice(None)] * 3
#     slices[time_axis] = slice(0, 50) 
    
#     data_subset = temp_mmap[tuple(slices)]
    
#     # Compute the mean image across the time axis
#     mean_img = np.mean(data_subset, axis=time_axis)
    
#     # Plot it
#     axes[i].imshow(mean_img, cmap='gray')
#     axes[i].set_title(f"Layout: {name_perm}\nShape: {shape_perm}")
#     axes[i].axis('off')
    
#     # Safely close the memmap to free memory for the next loop
#     if hasattr(temp_mmap, 'base') and hasattr(temp_mmap.base, 'close'):
#         temp_mmap.base.close()
#     del temp_mmap
#     gc.collect()

# plt.tight_layout()
# plt.show()

In [19]:
# denoised_memmap.flush()
# print(f"Saved denoised data to separate memmap: {denoised_memmap_path}")

# if hasattr(denoised_memmap, 'base') and hasattr(denoised_memmap.base, 'close'):
#     denoised_memmap.base.close()
# del denoised_memmap
# gc.collect()

In [20]:
# """
# Load Denoised TIFF and Verify Against Original ram_path
# ========================================================

# This script loads a denoised TIFF file from the R:/ drive and compares it
# visually with the original motion-corrected data stored in ram_path.

# CURRENT MODE: Verification (non-destructive)
# - Loads both datasets
# - Displays side-by-side visual comparison (mean images + correlation maps)
# - Saves both arrays separately
# - Does NOT overwrite original ram_path

# TO ENABLE OVERWRITING:
# - Uncomment the "OVERWRITE CODE" section at the end
# - Comment out the "SAVE DENOISED SEPARATELY" section
# - Re-run the script
# """

import glob
import gc
import os
import numpy as np
import tifffile
import matplotlib.pyplot as plt

# =============================================================================
# SECTION 1: LOAD DENOISED TIFF FROM R:/ DRIVE
# =============================================================================

print("Loading denoised TIFF from R:/ drive...")

denoised_tiff_pattern = os.path.join(r'R:/', '*denoised*.tif*')
denoised_files = glob.glob(denoised_tiff_pattern)

if not denoised_files:
    raise FileNotFoundError(f"No denoised TIFF found in R:/ matching *denoised*.tif*")

denoised_tiff_path = denoised_files[0]
print(f"Found denoised TIFF: {denoised_tiff_path}")

# Load multi-frame TIFF. Usually loads as (T, d1, d2) -> (35940, 18, 1108)
denoised_array = tifffile.imread(denoised_tiff_path).astype('float32')

# =============================================================================
# SECTION 2: LOAD ORIGINAL ram_path MEMMAP (d2, d1, T)
# =============================================================================

print(f"\nLoading original ram_path memmap: {ram_path}")

try:
    if 'video_mmap' in locals():
        safe_close_mmap(video_mmap)
except:
    pass
gc.collect()

# Define the discovered shape: (d2, d1, T)
expected_shape = (m_rig.shape[2], m_rig.shape[1], m_rig.shape[0])

video_mmap = np.memmap(
    ram_path,
    dtype='float32',
    mode='r',  
    shape=expected_shape,
    order='C'  
)
print(f"Loaded ram_path memmap. Shape: {video_mmap.shape}")

# =============================================================================
# SECTION 3: ALIGNMENT AND TEMPORAL STITCHING
# =============================================================================

# 1. Transpose the TIFF from (T, d1, d2) to (d2, d1, T)
# By reversing the axes (2, 1, 0), the time dimension is moved to the back.
print(f"Transposing denoised TIFF from {denoised_array.shape} to match layout...")
denoised_array = np.transpose(denoised_array, (2, 1, 0))

# 2. Temporal Stitching (Targeting the LAST axis for Time)
missing_frames = expected_shape[2] - denoised_array.shape[2]

if missing_frames > 0:
    pad_len = missing_frames // 2
    print(f"Stitching {pad_len} ORIGINAL frames to the start and end of axis 2...")
    
    seamless_array = np.empty(expected_shape, dtype=denoised_array.dtype)
    
    # Paste edges along the 3rd dimension
    seamless_array[:, :, :pad_len] = video_mmap[:, :, :pad_len]
    seamless_array[:, :, -pad_len:] = video_mmap[:, :, -pad_len:]
    
    # Paste denoised core into the middle of the 3rd dimension
    seamless_array[:, :, pad_len:-pad_len] = denoised_array
    
    denoised_array = seamless_array

# =============================================================================
# SECTION 4: SAVE DENOISED ARRAY TO DISK
# =============================================================================

print("\nSaving denoised array as backup memmap (separate file)...")

denoised_memmap_path = str(ram_path).replace('Green_rig', 'Green_rig_denoised')

denoised_memmap = np.memmap(
    denoised_memmap_path,
    dtype='float32',
    mode='w+', 
    shape=denoised_array.shape,
    order='C' 
)

# Stream denoised data into memmap across the time axis (axis 2)
chunk = 500  # Number of frames to write at once
T = expected_shape[2]

for t0 in range(0, T, chunk):
    t1 = min(t0 + chunk, T)
    denoised_memmap[:, :, t0:t1] = denoised_array[:, :, t0:t1]

denoised_memmap.flush()
print(f"Saved denoised data to separate memmap: {denoised_memmap_path}")

if hasattr(denoised_memmap, 'base') and hasattr(denoised_memmap.base, 'close'):
    denoised_memmap.base.close()
del denoised_memmap
gc.collect()

# =============================================================================
# SECTION 5: LOAD BOTH FROM DISK & PLOT VERIFICATION
# =============================================================================

print("\n=== VISUAL VERIFICATION (FROM DISK) ===")

denoised_mmap_verify = np.memmap(
    denoised_memmap_path,
    dtype='float32',
    mode='r',
    shape=expected_shape,
    order='C'
)

# Compute mean across the time axis (axis 2)
print("Computing mean images for both datasets directly from disk...\n")
orig_mean = np.mean(video_mmap, axis=2)
denoised_mean = np.mean(denoised_mmap_verify, axis=2)

# Create a 1x2 subplot figure (Side-by-side)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plotting the transpose (.T) of the means just so the 1108x18 image displays 
# horizontally on your monitor instead of vertically.
im1 = axes[0].imshow(orig_mean.T, cmap='gray')
axes[0].set_title('Original ram_path: Mean Image')
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

im2 = axes[1].imshow(denoised_mean.T, cmap='gray')
axes[1].set_title('Denoised .mmap: Mean Image')
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()

comparison_figure_path = os.path.join(rootpath, f'{unique_save_string}_denoised_comparison.png')
plt.savefig(comparison_figure_path, dpi=150, bbox_inches='tight')
plt.show()

print("\nComparison figure saved to:", comparison_figure_path)

# # =============================================================================
# # SECTION 6: SAVE DENOISED ARRAY SEPARATELY (VERIFICATION MODE)
# # =============================================================================

# print("\nSaving denoised array as backup memmap (separate file)...")

# # Create path for denoised memmap file by appending "_denoised" to ram_path
# denoised_memmap_path = str(ram_path).replace('.mmap', '_denoised.mmap')

# # Create new memmap with same format as original
# denoised_memmap = np.memmap(
#     denoised_memmap_path,
#     dtype='float32',
#     mode='w+',  # write/create mode
#     shape=denoised_array.shape,
#     order='F'   # Fortran/column-major order
# )

# # Stream denoised data into memmap in chunks (memory-efficient)
# chunk = 16  # frames per chunk
# T = denoised_array.shape[0]

# for t0 in range(0, T, chunk):
#     t1 = min(t0 + chunk, T)
#     denoised_memmap[t0:t1] = denoised_array[t0:t1]

# # Ensure data is flushed to disk
# denoised_memmap.flush()
# print(f"Saved denoised data to separate memmap: {denoised_memmap_path}")

# # Clean up memmap references
# if hasattr(denoised_memmap, 'base') and hasattr(denoised_memmap.base, 'close'):
#     denoised_memmap.base.close()
# del denoised_memmap

# print("\n=== ARRAYS SAVED SEPARATELY ===")
# print(f"Original:  {ram_path}")
# print(f"Denoised:  {denoised_memmap_path}")
# print("\nTo actually OVERWRITE ram_path with denoised data, modify this script to:")
# print("1. Remove the 'SAVE DENOISED SEPARATELY' section above")
# print("2. Uncomment the 'OVERWRITE CODE' section below")

# =============================================================================
# SECTION 7: OVERWRITE CODE (COMMENTED OUT - ENABLE LATER)
# =============================================================================
# Uncomment the section below to actually overwrite ram_path with denoised data.
# WARNING: This is destructive and will replace the original motion-corrected data.
#
# To enable:
# 1. Uncomment all lines between the triple-quoted markers below
# 2. Comment out or remove SECTION 6 above
# 3. Re-run the script

# """
# print("Opening ram_path memmap in read-write mode...")

# # Close any previous references to ram_path
# try:
#     if 'video_mmap' in locals():
#         safe_close_mmap(video_mmap)
# except:
#     pass

# gc.collect()

# # Open memmap in read-write mode and overwrite with denoised data
# dst = np.memmap(
#     ram_path,
#     dtype='float32',
#     mode='r+',  # read-write mode (file must already exist)
#     shape=(m_rig.shape[0], m_rig.shape[2], m_rig.shape[1]),  # (T, W, H)
#     order='F'   # Must use same order as original
# )

# print("Overwriting ram_path memmap with denoised data...")

# # Stream the denoised data into the memmap in chunks
# chunk = 16  # frames per chunk
# T = denoised_array.shape[0]

# for t0 in range(0, T, chunk):
#     t1 = min(t0 + chunk, T)
#     dst[t0:t1] = denoised_array[t0:t1]

# # Ensure data is written to disk
# dst.flush()
# print(f"Successfully overwrote ram_path with denoised data. Shape: {dst.shape}, Order: F")

# # Clean up memmap references
# if hasattr(dst, 'base') and hasattr(dst.base, 'close'):
#     dst.base.close()
# del dst
# gc.collect()

# print("Denoised data is now in ram_path. Ready for downstream analysis.")
# """


Loading denoised TIFF from R:/ drive...
Found denoised TIFF: R:/temp_processing_stack_denoised.tif

Loading original ram_path memmap: R:\FOV1_T4_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_108000.mmap
Loaded ram_path memmap. Shape: (1108, 18, 108000)
Transposing denoised TIFF from (107940, 18, 1108) to match layout...
Stitching 30 ORIGINAL frames to the start and end of axis 2...

Saving denoised array as backup memmap (separate file)...
Saved denoised data to separate memmap: R:\FOV1_T4_Green_rig_denoised__d1_18_d2_1108_d3_1_order_C_frames_108000.mmap

=== VISUAL VERIFICATION (FROM DISK) ===
Computing mean images for both datasets directly from disk...


Comparison figure saved to: D:\pAce\Analysis\BKV009-20260703-FOV1_T4_denoised_comparison.png


In [21]:
ram_path

'R:\\FOV1_T4_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_108000.mmap'

In [22]:
# img.shape

In [23]:
# denoised_mean.T.shape

In [24]:
# ram_path = r'R:\FOV1_T14_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_36000.mmap'

In [25]:
# ram_path

In [26]:
img_corr = cmap_data
img2 = denoised_mean.T
summary_images2 = np.stack([img2, img2, img_corr], axis=0).astype(np.float32)
#cm.movie(summary_images).save(fname[:-5]+'_summary_images.tif')

plt.imshow(summary_images2[0], cmap='gray')
plt.axis('off')
#plt.savefig(fname[:-4]+'_mean.tif', format='tif', bbox_inches='tight', pad_inches=0)


plt.imshow(summary_images2[2], cmap='gray')
plt.axis('off')
#plt.savefig(fname[:-4]+'_corr.tif', format='tif', bbox_inches='tight', pad_inches=0)

img2 = summary_images2.transpose([1, 2, 0])


print(fname[:-4]+'_corr.tif')
height, width = img2.shape[:2]
print(img2.shape)

# --------------------------------------------------------------
# Extract channels like MATLAB
# --------------------------------------------------------------
R = img2[:, :, 0]
B = img2[:, :, 2]

# --------------------------------------------------------------
# MATLAB-style normalization (mat2gray + uint8)
# --------------------------------------------------------------
def normalize_like_matlab(x):
    x = x.astype(np.float64)
    mn = x.min()
    mx = x.max()
    x = (x - mn) / (mx - mn + 1e-12)

    # MATLAB uint8 applies rounding, not floor
    x = np.round(255 * x).astype(np.uint8)
    return x

R_norm = normalize_like_matlab(R)
B_norm = normalize_like_matlab(B)

# --------------------------------------------------------------
# Build MATLAB-equivalent RGB (R,R,B)
# --------------------------------------------------------------
rgb2 = np.stack([R_norm, R_norm, B_norm], axis=2).astype(np.uint8)

# --------------------------------------------------------------
# Save as PNG/TIF (MATLAB-compatible pixel data)
# --------------------------------------------------------------
outname =  rootpath + unique_save_string + ".tif"
Image.fromarray(rgb2).save(outname)

print("Saved:", outname)
img2 = rgb2.copy()
img = img2.copy()

D:\pAce\BKV009\20260703\FOV1_T4\FOV1_T4_Green.d_corr.tif
(18, 1108, 3)
Saved: D:\pAce\Analysis\BKV009-20260703-FOV1_T4.tif


In [27]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Create the 512x512 Canvas
# ---------------------------------------------------------
# Ensure canvas matches the data type of your image (likely uint8 or float32)
canvas_size = 512
canvas = np.zeros((canvas_size, canvas_size, 3), dtype=img.dtype)

h, w, c = img.shape
chunk_width = canvas_size - 20  # Leave a 10px margin on the left/right
overlap = 40                    # Number of pixels to overlap between strips
stride = chunk_width - overlap  # How far to step forward each time

y_start = 50                    # Start 50 pixels down from the top
y_spacing = 60                  # Spacing between strips

# ---------------------------------------------------------
# 2. Slice the 18x1108 image and stack it onto the canvas
# ---------------------------------------------------------
print(f"Tiling image of shape {img.shape} onto a 512x512 canvas with {overlap}px overlap...")

chunks = []
current_y = y_start

for x in range(0, w, stride):
    # Ensure we don't grab more pixels than are left in the image
    c_w = min(chunk_width, w - x)
    
    # Slice a chunk of the image
    chunk = img[:, x : x + c_w, :]
    
    # Paste it onto the canvas
    canvas[current_y : current_y + h, 10 : 10 + c_w, :] = chunk
    
    # Define the "core" x-region for this chunk to prevent double counting in overlaps.
    # The first chunk owns the first half of its overlap, the next chunk owns the second half.
    core_x_start = 10 + (overlap // 2 if x > 0 else 0)
    core_x_end = 10 + c_w - (overlap // 2 if x + stride < w else 0)
    
    # Save the coordinates to map detections back and filter invalid ones
    chunks.append({
        'orig_x_start': x,
        'canvas_y_start': current_y,
        'canvas_y_end': current_y + h,
        'canvas_x_start': 10,
        'canvas_x_end': 10 + c_w,
        'core_x_start': core_x_start,
        'core_x_end': core_x_end,
        'chunk_width': c_w
    })
    
    current_y += y_spacing

# ---------------------------------------------------------
# 3. Run Inference on the Composite Canvas
# ---------------------------------------------------------
print("Running Mask R-CNN inference on tiled canvas...")
r = utils.mrcnn_inference(canvas, size_range=[0, 40], weights_path=weights_path, display_result=False)

# ---------------------------------------------------------
# 4. Filter Invalid, Duplicate, and Transform Detections
# ---------------------------------------------------------
valid_orig_masks = []  # To hold the masks transformed back to 18x1108
valid_orig_rois = []   # To hold the bounding boxes transformed back
valid_scores = []
valid_class_ids = []

if r['masks'].shape[-1] > 0:
    n_detections = r['masks'].shape[-1]
    
    for i in range(n_detections):
        # r['rois'] is typically [y1, x1, y2, x2]
        y1, x1, y2, x2 = r['rois'][i]
        
        # Calculate the center point of the detection
        cy = (y1 + y2) / 2.0
        cx = (x1 + x2) / 2.0
        
        # Check if the center falls strictly inside the "core" region of any chunk
        is_valid = False
        assigned_chunk = None
        for chunk in chunks:
            if (chunk['canvas_y_start'] <= cy <= chunk['canvas_y_end']) and \
               (chunk['core_x_start'] <= cx <= chunk['core_x_end']):
                is_valid = True
                assigned_chunk = chunk
                break
                
        if is_valid:
            valid_scores.append(r['scores'][i])
            valid_class_ids.append(r['class_ids'][i])
            
            # --- Transform ROIs back to original coordinates ---
            orig_y1 = y1 - assigned_chunk['canvas_y_start']
            orig_y2 = y2 - assigned_chunk['canvas_y_start']
            orig_x1 = x1 - assigned_chunk['canvas_x_start'] + assigned_chunk['orig_x_start']
            orig_x2 = x2 - assigned_chunk['canvas_x_start'] + assigned_chunk['orig_x_start']
            
            # Clip to image boundaries to prevent out-of-bounds indices
            orig_y1 = max(0, min(h, orig_y1))
            orig_y2 = max(0, min(h, orig_y2))
            orig_x1 = max(0, min(w, orig_x1))
            orig_x2 = max(0, min(w, orig_x2))
            
            valid_orig_rois.append([orig_y1, orig_x1, orig_y2, orig_x2])
            
            # --- Transform Masks back to original coordinates ---
            # Extract the valid mask using the assigned chunk's canvas coordinates
            mask_chunk = r['masks'][
                assigned_chunk['canvas_y_start'] : assigned_chunk['canvas_y_end'],
                assigned_chunk['canvas_x_start'] : assigned_chunk['canvas_x_end'],
                i
            ]
            
            # Paste it into an empty mask matching the original image dimensions
            orig_mask = np.zeros((h, w), dtype=bool)
            orig_mask[:, assigned_chunk['orig_x_start'] : assigned_chunk['orig_x_start'] + assigned_chunk['chunk_width']] = mask_chunk
            valid_orig_masks.append(orig_mask)

# --- Compile the Final Reconstructed Dictionary ---
r_reconstructed = {
    'rois': np.array(valid_orig_rois, dtype=np.int32) if len(valid_orig_rois) > 0 else np.empty((0, 4), dtype=np.int32),
    'class_ids': np.array(valid_class_ids, dtype=np.int32),
    'scores': np.array(valid_scores, dtype=np.float32),
    'masks': np.stack(valid_orig_masks, axis=-1) if len(valid_orig_masks) > 0 else np.empty((h, w, 0), dtype=bool)
}

print("\n--- Reconstructed Results ---")
print("r_reconstructed keys:", r_reconstructed.keys())
print("r_reconstructed['rois'] shape:", r_reconstructed['rois'].shape)
print("r_reconstructed['masks'] shape:", r_reconstructed['masks'].shape)
print("r_reconstructed['scores']:", r_reconstructed['scores'])


# ---------------------------------------------------------
# 5. Summarize and Plot the Tiled Canvas Results
# ---------------------------------------------------------
# We will use the r_reconstructed object for our downstream checks to ensure it works!
n_valid = r_reconstructed['masks'].shape[-1]

if n_valid > 0:
    print(f"\nSuccess! Found {n_valid} valid objects strictly within the image regions.")
    # Show the canvas detections by projecting the reconstructed masks back onto a visual sum
    # (Optional: If you still wanted to see the canvas sum, we skip it here to focus on the real data)
else:
    print("\nNo valid objects detected in the continuous image regions.")

fig, axs = plt.subplots(1, 2, figsize=(14, 7))

# Show the canvas we built
axs[0].imshow(canvas)
axs[0].set_title('Composite 512x512 Input Canvas')
axs[0].axis('off')

# Show original image with reconstructed masks overlayed directly as a sum
axs[1].imshow(img, aspect="equal")  # Background
if n_valid > 0:
    masks_sum = r_reconstructed['masks'].sum(axis=-1)
    axs[1].imshow(masks_sum, cmap='jet', alpha=0.5, aspect="equal")  # Overlay valid masks
axs[1].set_title(f'Mask R-CNN Reconstructed Valid Masks (n={n_valid})')
axs[1].axis('off')

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 6. Reconstruct and Plot Original Image with Contours
# ---------------------------------------------------------
print("Generating reconstructed original image with contours...")

fig2, ax2 = plt.subplots(figsize=(18, 4))

# Show original image (aspect="equal" to maintain true pixel proportions)
ax2.imshow(img, aspect="equal") 
ax2.set_title(f'Original Image ({h}x{w}) Reconstructed with {n_valid} ROI Contours')
ax2.axis('off')

# Overlay each valid mask from our reconstructed dictionary as a contour line
if n_valid > 0:
    for i in range(n_valid):
        mask = r_reconstructed['masks'][:, :, i]
        if np.any(mask):
            ax2.contour(mask, levels=[0.5], colors='red', linewidths=1)

plt.tight_layout()
plt.show()


r=r_reconstructed
ROIs = r['masks'].transpose([2, 0, 1])
Coords = r['rois']
#cm.movie(ROIs).save(fname[:-4]+'newmrcnn_ROIs.hdf5')



fig, axs = plt.subplots(2, 1, figsize=(10, 15))
axs[0].imshow(summary_images2[1])
axs[1].imshow(ROIs.sum(0))
axs[0].set_title('mean image')
axs[1].set_title('masks')


#save ROIs as npy array
np.save(fname[:-6]+'newmrcnn_ROIs.npy', ROIs)
print("Saved ROIs as npy array:", fname[:-6]+'newmrcnn_ROIs.npy')

Tiling image of shape (18, 1108, 3) onto a 512x512 canvas with 40px overlap...
Running Mask R-CNN inference on tiled canvas...

Configurations:
BACKBONE                       resnet50
BACKBONE_STRIDES               [4, 8, 16, 32, 64]
BATCH_SIZE                     1
BBOX_STD_DEV                   [0.1 0.1 0.2 0.2]
COMPUTE_BACKBONE_SHAPE         None
DETECTION_MAX_INSTANCES        200
DETECTION_MIN_CONFIDENCE       0
DETECTION_NMS_THRESHOLD        0.3
FPN_CLASSIF_FC_LAYERS_SIZE     1024
GPU_COUNT                      1
GRADIENT_CLIP_NORM             5.0
IMAGES_PER_GPU                 1
IMAGE_CHANNEL_COUNT            3
IMAGE_MAX_DIM                  512
IMAGE_META_SIZE                14
IMAGE_MIN_DIM                  512
IMAGE_MIN_SCALE                0
IMAGE_RESIZE_MODE              crop
IMAGE_SHAPE                    [512 512   3]
LEARNING_MOMENTUM              0.9
LEARNING_RATE                  0.001
LOSS_WEIGHTS                   {'rpn_class_loss': 1.0, 'rpn_bbox_loss': 1.0, 'mrcnn_c

     6217534 [deprecation.py:            new_func():554][26180] From c:\Users\strenglab\anaconda3\envs\caiman\lib\site-packages\tensorflow\python\util\deprecation.py:629: calling map_fn_v2 (from tensorflow.python.ops.map_fn) with dtype is deprecated and will be removed in a future version.
Instructions for updating:
Use fn_output_signature instead


Processing 1 images
image                    shape: (512, 512, 3)         min:    0.00000  max:  255.00000  uint8
molded_images            shape: (1, 512, 512, 3)      min:  -91.11000  max:  168.24000  float64
image_metas              shape: (1, 14)               min:    0.00000  max:  512.00000  int32
anchors                  shape: (1, 65472, 4)         min:   -0.04428  max:    1.01297  float32


c:\Users\strenglab\anaconda3\envs\caiman\lib\site-packages\keras\engine\training_v1.py:2356: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


MADE FIGURE

--- Reconstructed Results ---
r_reconstructed keys: dict_keys(['rois', 'class_ids', 'scores', 'masks'])
r_reconstructed['rois'] shape: (7, 4)
r_reconstructed['masks'] shape: (18, 1108, 7)
r_reconstructed['scores']: [0.99689555 0.9917635  0.990213   0.85201037 0.7745484  0.7275864
 0.5054345 ]

Success! Found 7 valid objects strictly within the image regions.
Generating reconstructed original image with contours...
Saved ROIs as npy array: D:\pAce\BKV009\20260703\FOV1_T4\FOV1_T4_Greennewmrcnn_ROIs.npy


In [28]:
# print(r.keys())
# print(r["rois"][0].shape)
# print(r["class_ids"])
# print(r["scores"])
# print(r["masks"].shape)


In [29]:
# print(r["rois"])

In [30]:


# ##
# print("Running Mask R-CNN inference...")
# #download_model('mask_rcnn')
# #ROIs, r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=True)
# r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=False)
# ROIs = r['masks'].transpose([2, 0, 1])
# Coords = r['rois']
# #cm.movie(ROIs).save(fname[:-4]+'newmrcnn_ROIs.hdf5')

# fig, axs = plt.subplots(1, 2)
# axs[0].imshow(summary_images[1])
# axs[1].imshow(ROIs.sum(0))
# axs[0].set_title('mean image')
# axs[1].set_title('masks')
# #plt.savefig(fname[:-6] + 'newmrcnn_ROIs.png', format='png', bbox_inches='tight', pad_inches=0)
# # Save the figure and close the plot   

# #save ROIs as npy array
# #np.save(fname[:-4]+'newmrcnn_ROIs.npy', ROIs)
# #print("Saved ROIs as npy array:", fname[:-4]+'newmrcnn_ROIs.npy')


In [31]:
print(ram_path)

R:\FOV1_T4_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_108000.mmap


In [32]:
# import numpy as np
# import cv2
# import os

# # ---------------------------------------------------------
# # 1. Configuration from your filename metadata
# # ---------------------------------------------------------
# # Path variable as requested
# # ram_path = r"R:\FOV1_T14_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_36000.mmap"

# d1 = 18       # Height
# d2 = 1108     # Width
# d3 = 1        # Channels
# frames = 36000
# order = 'F'   # C-style memory layout

# # Expected shape for standard video frame indexing: (Frames, Height, Width)
# video_shape = (frames, d1, d2)

# # ---------------------------------------------------------
# # 2. Load Memory-Mapped Video
# # ---------------------------------------------------------
# print(f"Mapping video from: {ram_path}")
# if not os.path.exists(ram_path):
#     raise FileNotFoundError(f"Could not find the file at {ram_path}")

# # Load in read-only mode ('r') so we don't accidentally modify the data
# video_mmap = np.memmap(
#     ram_path, 
#     dtype=np.float32, 
#     mode='r', 
#     shape=video_shape, 
#     order=order
# )

# print(f"Successfully mapped array with shape: {video_mmap.shape}")

# # ---------------------------------------------------------
# # 3. Video Playback Loop (OpenCV Window)
# # ---------------------------------------------------------
# window_name = "CaImAn Memory Map Player (Press 'q' to Quit)"
# cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

# print("Starting playback. Click on the video window and press 'q' to stop.")

# for f in range(frames):
#     # 1. Pull a single frame: shape is (18, 1108)
#     frame = video_mmap[f, :, :]
    
#     # 2. Normalize data to 0-255 range for image display if it's float32 raw data
#     f_min, f_max = frame.min(), frame.max()
#     if f_max - f_min > 0:
#         frame_display = ((frame - f_min) / (f_max - f_min) * 255).astype(np.uint8)
#     else:
#         frame_display = np.zeros_like(frame, dtype=np.uint8)
#     # 3. STRETCH THE WINDOW VERTICALLY FOR VISUALIZATION
#     # Native 18x1108 is too tiny to see. Let's scale the height up to 200 pixels.
#     display_height = 18
#     display_width = 1108
#     frame_resized = cv2.resize(frame_display, (display_width, display_height), interpolation=cv2.INTER_NEAREST)
    
#     # 4. Show the frame
#     cv2.imshow(window_name, frame_resized)
    
#     # 5. Frame rate control: wait 1 millisecond between frames
#     # This also listens for keyboard input. If 'q' is pressed, break loop.
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         print("Playback stopped by user.")
#         break

# # Clean up window and safely unmap reference
# cv2.destroyAllWindows()
# if hasattr(video_mmap, 'base') and hasattr(video_mmap.base, 'close'):
#     video_mmap.base.close()
# print("Playback window closed safely.")

In [33]:

###NEW SECTION FOR ROI COORDINATE EXTRACTION
cell_centers = [((y1 + y2) // 2, (x1 + x2) // 2) for (y1, x1, y2, x2) in Coords]
cell_centers = np.array(cell_centers)
print("Cell centers:", cell_centers)    
#display the cell centers on the image
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.imshow(img, cmap='gray') # Display the image
# ax.scatter(cell_centers[:, 1], cell_centers[:, 0], color='red') # Display the cell centers
# ax.set_title('Cell centers')    # Set the title of the plot
#plt.savefig(fname[:-4] + '_cell_centers.png', format='png', bbox_inches='tight', pad_inches=0)
 # Save the figure and close the plot     

# Save to a file
save_path = fname[:-6] + '_cell_centers.npy'
np.save(save_path, cell_centers)

print(f"Cell centers saved to {save_path}")

#check if ROIS are empty and if so skip and save error
if ROIs.shape[0] == 0:
    print("No ROIs found.")
    raise ValueError("No ROIs detected, skipping further analysis for this trial.")
else:
    print(f"Found {ROIs.shape[0]} ROIs.")


cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False, maxtasksperchild=1)


Cell centers: [[   9  519]
 [   7  480]
 [   8  653]
 [  11  671]
 [   5  569]
 [   7  596]
 [   4 1105]]
Cell centers saved to D:\pAce\BKV009\20260703\FOV1_T4\FOV1_T4_Green_cell_centers.npy
Found 7 ROIs.


In [34]:
ram_path


'R:\\FOV1_T4_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_108000.mmap'

In [35]:
ram_path = r'R:\FOV1_T14_Green_rig_denoised__d1_18_d2_1108_d3_1_order_C_frames_36000.mmap'

In [36]:
ram_path = r'R:\FOV1_T14_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_36000.mmap'

In [37]:
ram_path

'R:\\FOV1_T14_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_36000.mmap'

In [39]:
"""
Plotting utilities for visualizing debug_info from volspike_debug()

Use these functions in your interactive script after calling volspike_debug() on a single ROI:

    from caiman.source_extraction.volpy.spikepursuit_debug import volspike_debug
    output, debug_info = volspike_debug(pars, return_debug_info=True)
    
    from plot_debug_info import *
    plot_intermediate_t0(debug_info, output)
    plot_data_hp(debug_info)
    plot_traces_comparison(debug_info, output)
    plot_iteration_progression(debug_info)
    etc.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec


def plot_intermediate_t0(debug_info, output):
    """Plot the initial trace (t0) before background removal"""
    t0 = debug_info['intermediate_t0']
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    
    # Raw trace
    axes[0].plot(t0, linewidth=0.5, color='steelblue')
    axes[0].set_xlabel('Time (frames)')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title('Intermediate t0: Initial Trace (from ROI mean, before background removal)')
    axes[0].grid(True, alpha=0.3)
    
    # Histogram
    axes[1].hist(t0, bins=100, color='steelblue', alpha=0.7, edgecolor='black')
    axes[1].set_xlabel('Amplitude')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Distribution of t0 values')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print(f"t0 statistics:")
    print(f"  Shape: {t0.shape}")
    print(f"  Min: {np.min(t0):.4f}, Max: {np.max(t0):.4f}")
    print(f"  Mean: {np.mean(t0):.4f}, Std: {np.std(t0):.4f}")


def plot_t0_no_bg(debug_info, output):
    """Plot the trace after background removal"""
    t0_no_bg = debug_info['t0_no_bg']
    t0 = debug_info['intermediate_t0']
    
    fig, axes = plt.subplots(3, 1, figsize=(12, 8))
    
    # Before BG removal
    axes[0].plot(t0, linewidth=0.5, color='steelblue', label='Before BG removal')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title('t0: Before Background Removal')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    # After BG removal
    axes[1].plot(t0_no_bg, linewidth=0.5, color='coral', label='After BG removal')
    axes[1].set_ylabel('Amplitude')
    axes[1].set_title('t0 no_bg: After Background Removal')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    
    # Difference (what was removed)
    bg_removed = t0 - t0_no_bg
    axes[2].plot(bg_removed, linewidth=0.5, color='green', label='Background component')
    axes[2].set_xlabel('Time (frames)')
    axes[2].set_ylabel('Amplitude')
    axes[2].set_title('Background Component Removed')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"Background removal statistics:")
    print(f"  t0 (before): std={np.std(t0):.4f}")
    print(f"  t0_no_bg (after): std={np.std(t0_no_bg):.4f}")
    print(f"  BG component: std={np.std(bg_removed):.4f}")


def plot_data_hp(debug_info):
    """
    Plot the high-pass filtered data from the context region around ROI.
    
    data_hp has shape (T, N_pixels) where:
    - T = number of frames/timepoints
    - N_pixels = all pixels in the extracted context region around the ROI
    
    We can view it multiple ways:
    1. As a heatmap (time × pixels)
    2. As individual pixel traces
    3. As spatially organized grid (if we know the context region dimensions)
    """
    data_hp = debug_info['data_hp']
    T, N_pixels = data_hp.shape
    
    print(f"data_hp statistics:")
    print(f"  Shape: {data_hp.shape} (T={T} frames, N_pixels={N_pixels})")
    print(f"  Contains: {N_pixels} different pixel traces, each with {T} timepoints")
    print(f"  Min: {np.min(data_hp):.4f}, Max: {np.max(data_hp):.4f}")
    print(f"  Mean per pixel: {np.mean(data_hp, axis=0)}")
    
    fig = plt.figure(figsize=(14, 10))
    gs = GridSpec(3, 2, figure=fig)
    
    # Heatmap view
    ax1 = fig.add_subplot(gs[0, :])
    im = ax1.imshow(data_hp.T, aspect='auto', cmap='viridis', interpolation='nearest')
    ax1.set_xlabel('Time (frames)')
    ax1.set_ylabel('Pixel index')
    ax1.set_title(f'data_hp Heatmap: {N_pixels} pixels × {T} frames')
    plt.colorbar(im, ax=ax1, label='Amplitude')
    
    # Mean trace across all pixels
    ax2 = fig.add_subplot(gs[1, 0])
    mean_trace = np.mean(data_hp, axis=1)
    ax2.plot(mean_trace, linewidth=0.8, color='steelblue')
    ax2.set_xlabel('Time (frames)')
    ax2.set_ylabel('Mean Amplitude')
    ax2.set_title('Mean across all pixels')
    ax2.grid(True, alpha=0.3)
    
    # Standard deviation across pixels at each timepoint
    ax3 = fig.add_subplot(gs[1, 1])
    std_trace = np.std(data_hp, axis=1)
    ax3.plot(std_trace, linewidth=0.8, color='coral')
    ax3.set_xlabel('Time (frames)')
    ax3.set_ylabel('Std Dev')
    ax3.set_title('Spatial variability at each timepoint')
    ax3.grid(True, alpha=0.3)
    
    # Distribution of pixel traces
    ax4 = fig.add_subplot(gs[2, 0])
    ax4.hist(data_hp.ravel(), bins=100, color='steelblue', alpha=0.7, edgecolor='black')
    ax4.set_xlabel('Amplitude')
    ax4.set_ylabel('Count')
    ax4.set_title('Distribution of all pixel values')
    ax4.grid(True, alpha=0.3, axis='y')
    
    # A few example individual pixel traces
    ax5 = fig.add_subplot(gs[2, 1])
    n_examples = min(10, N_pixels)
    indices = np.linspace(0, N_pixels-1, n_examples, dtype=int)
    for idx in indices:
        ax5.plot(data_hp[:, idx], linewidth=0.5, alpha=0.6, label=f'Pixel {idx}')
    ax5.set_xlabel('Time (frames)')
    ax5.set_ylabel('Amplitude')
    ax5.set_title(f'Example {n_examples} pixel traces from context region')
    ax5.grid(True, alpha=0.3)
    ax5.legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()


def plot_background_components(debug_info):
    """
    Plot the background principal components (Ub from SVD)
    
    Ub has shape (T, nPC_bg) where:
    - T = number of frames
    - nPC_bg = number of principal components (usually 8)
    
    Each column is a temporal basis vector for the background
    """
    Ub = debug_info['Ub']
    T, nPC = Ub.shape
    
    print(f"Ub (background components) statistics:")
    print(f"  Shape: {Ub.shape} (T={T} frames, {nPC} principal components)")
    
    fig, axes = plt.subplots(nPC, 1, figsize=(12, 10))
    if nPC == 1:
        axes = [axes]
    
    for i in range(nPC):
        axes[i].plot(Ub[:, i], linewidth=0.8, color='steelblue')
        axes[i].set_ylabel(f'PC {i+1}')
        axes[i].set_title(f'Background Principal Component {i+1}')
        axes[i].grid(True, alpha=0.3)
        
        # Show statistics
        print(f"  PC{i+1}: min={np.min(Ub[:, i]):.4f}, max={np.max(Ub[:, i]):.4f}, std={np.std(Ub[:, i]):.4f}")
    
    axes[-1].set_xlabel('Time (frames)')
    plt.tight_layout()
    plt.show()


def plot_iteration_progression(debug_info, output):
    """
    Plot how the signal and spike detection changed across iterations
    """
    iteration_details = debug_info['iteration_details']
    n_iter = len(iteration_details)
    
    print(f"\nIteration progression ({n_iter} iterations):")
    
    # Summary of spikes per iteration
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    spike_counts = [it['num_spikes'] for it in iteration_details]
    thresholds = [it['threshold'] for it in iteration_details]
    shrinkage_factors = [it['shrinkage_factor'] for it in iteration_details]
    iterations = list(range(len(iteration_details)))
    
    # Spike counts per iteration
    axes[0, 0].plot(iterations, spike_counts, 'o-', linewidth=2, markersize=8, color='steelblue')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Number of Spikes')
    axes[0, 0].set_title('Spike Count Progression')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Thresholds per iteration
    axes[0, 1].plot(iterations, thresholds, 'o-', linewidth=2, markersize=8, color='coral')
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].set_ylabel('Threshold')
    axes[0, 1].set_title('Spike Detection Threshold Progression')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Shrinkage factors per iteration
    axes[1, 0].plot(iterations, shrinkage_factors, 'o-', linewidth=2, markersize=8, color='green')
    axes[1, 0].set_xlabel('Iteration')
    axes[1, 0].set_ylabel('Shrinkage Factor')
    axes[1, 0].set_title('Amplitude Correction (Shrinkage Factor)')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Final spike times from last iteration
    final_spikes = iteration_details[-1]['spikes_final']
    axes[1, 1].scatter(final_spikes, np.ones_like(final_spikes), alpha=0.5, s=10, color='steelblue')
    axes[1, 1].set_xlabel('Time (frames)')
    axes[1, 1].set_ylabel('Spike')
    axes[1, 1].set_title(f'Final Spike Times ({len(final_spikes)} spikes)')
    axes[1, 1].set_ylim([0.5, 1.5])
    axes[1, 1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    for i, it in enumerate(iteration_details):
        print(f"  Iteration {i}: {it['num_spikes']:3d} spikes, threshold={it['threshold']:8.4f}, shrinkage={it['shrinkage_factor']:.4f}")


def plot_iteration_details_single(debug_info, iteration_idx):
    """
    Plot detailed traces for a specific iteration
    
    Shows: trace after reconstruction, after BG removal, and reconstructed/subthreshold
    """
    if 'iteration_details' not in debug_info:
        print("No iteration details in debug_info")
        return
    
    iteration_details = debug_info['iteration_details']
    if iteration_idx >= len(iteration_details):
        print(f"Iteration {iteration_idx} not found (only {len(iteration_details)} iterations)")
        return
    
    it = iteration_details[iteration_idx]
    t_after_recon = it.get('t_after_recon')
    t_after_bg = it.get('t_after_bg_removal')
    t_final = it.get('t_final')
    t_rec_final = it.get('t_rec_final')
    spikes_final = it.get('spikes_final')
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    
    T = len(t_final)
    t_axis = np.arange(T)
    
    # Trace progression through iteration
    if t_after_recon is not None:
        axes[0].plot(t_axis, t_after_recon, linewidth=0.5, alpha=0.7, label='After reconstruction')
    if t_after_bg is not None:
        axes[0].plot(t_axis, t_after_bg, linewidth=0.5, alpha=0.7, label='After BG removal')
    axes[0].plot(t_axis, t_final, linewidth=0.8, label='Final trace', color='steelblue')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title(f'Iteration {iteration_idx}: Trace Evolution')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    # Reconstructed spikes
    axes[1].plot(t_axis, t_final, linewidth=0.8, color='steelblue', label='Final trace')
    axes[1].plot(t_axis, t_rec_final, linewidth=1.5, color='red', label='Reconstructed spikes')
    if spikes_final is not None and len(spikes_final) > 0:
        axes[1].scatter(spikes_final, t_final[spikes_final], color='green', s=30, marker='x', linewidths=2, label='Spike times')
    axes[1].set_ylabel('Amplitude')
    axes[1].set_title(f'Iteration {iteration_idx}: Detected Spikes')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    
    # Subthreshold component
    subthreshold = t_final - t_rec_final
    axes[2].plot(t_axis, subthreshold, linewidth=0.8, color='coral')
    axes[2].set_xlabel('Time (frames)')
    axes[2].set_ylabel('Amplitude')
    axes[2].set_title(f'Iteration {iteration_idx}: Subthreshold Activity')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Iteration {iteration_idx} details:")
    print(f"  Spikes detected: {len(spikes_final)}")
    print(f"  Threshold: {it.get('threshold', 'N/A')}")
    print(f"  Shrinkage factor: {it.get('shrinkage_factor', 'N/A')}")


def plot_final_results(output, debug_info):
    """
    Plot the final results alongside input/processing data
    """
    fig = plt.figure(figsize=(16, 10))
    gs = GridSpec(4, 2, figure=fig, hspace=0.3, wspace=0.3)
    
    # Initial trace (from debug_info)
    ax1 = fig.add_subplot(gs[0, :])
    t0 = debug_info['intermediate_t0']
    ax1.plot(t0, linewidth=0.5, color='lightgray', label='Initial t0')
    ax1.set_ylabel('Amplitude')
    ax1.set_title('Signal Processing: From Initial Trace to Final Results')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Final trace
    ax2 = fig.add_subplot(gs[1, :])
    t_final = output['t']
    t_rec = output['t_rec']
    spikes = output['spikes']
    ax2.plot(t_final, linewidth=0.8, color='steelblue', label='Final trace (t)')
    ax2.plot(t_rec, linewidth=1, color='red', label='Reconstructed (t_rec)')
    if len(spikes) > 0:
        ax2.scatter(spikes, t_final[spikes], color='green', s=30, marker='x', linewidths=2, label='Detected spikes')
    ax2.set_ylabel('Amplitude')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # dF/F
    ax3 = fig.add_subplot(gs[2, :])
    dFF = output['dFF']
    ax3.plot(dFF, linewidth=0.8, color='steelblue')
    ax3.set_ylabel('dF/F')
    ax3.set_title('ΔF/F (Normalized Signal)')
    ax3.grid(True, alpha=0.3)
    
    # Summary statistics
    ax4 = fig.add_subplot(gs[3, 0])
    ax4.axis('off')
    stats_text = f"""
    FINAL RESULTS SUMMARY
    
    Total spikes detected: {len(spikes)}
    Low spikes warning: {output['low_spikes']}
    
    Signal statistics:
    • Min: {np.min(t_final):.4f}
    • Max: {np.max(t_final):.4f}
    • Mean: {np.mean(t_final):.4f}
    • Std: {np.std(t_final):.4f}
    
    Quality metrics:
    • SNR: {output['snr']:.3f}
    • Threshold: {output['thresh']:.4f}
    • Locality passed: {output['locality']}
    • Polarity: {output.get('polarity', 'N/A')}
    """
    ax4.text(0.1, 0.5, stats_text, fontfamily='monospace', fontsize=10, verticalalignment='center')
    
    # Spike times histogram
    ax5 = fig.add_subplot(gs[3, 1])
    if len(spikes) > 0:
        ax5.hist(spikes, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
        ax5.set_xlabel('Time (frames)')
        ax5.set_ylabel('Spike count')
    ax5.set_title('Distribution of Spike Times')
    ax5.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()


# =============================================================================
# QUICK START - Execute these in your interactive script after volspike_debug()
# =============================================================================

def plot_all_debug_info(debug_info, output):
    """Convenience function to plot all debug information"""
    print("Plotting all debug information...\n")
    
    plot_intermediate_t0(debug_info, output)
    print("\n" + "="*60 + "\n")
    
    plot_t0_no_bg(debug_info, output)
    print("\n" + "="*60 + "\n")
    
    plot_data_hp(debug_info)
    print("\n" + "="*60 + "\n")
    
    plot_background_components(debug_info)
    print("\n" + "="*60 + "\n")
    
    plot_iteration_progression(debug_info, output)
    print("\n" + "="*60 + "\n")
    
    plot_final_results(output, debug_info)


# =============================================================================
# EXAMPLE INTERACTIVE USAGE
# =============================================================================
"""
from caiman.source_extraction.volpy.spikepursuit_debug import volspike_debug
from plot_debug_info import *

# Run debug version on single ROI
output, debug_info = volspike_debug(pars, return_debug_info=True)

# Choose what to plot:
plot_intermediate_t0(debug_info, output)           # Initial trace before BG removal
plot_t0_no_bg(debug_info, output)                  # Before/after background removal
plot_data_hp(debug_info)                           # All pixel traces in context region
plot_background_components(debug_info)             # SVD background components
plot_iteration_progression(debug_info, output)     # How spikes/threshold changed per iteration
plot_iteration_details_single(debug_info, 0)       # Detailed traces for iteration 0
plot_final_results(output, debug_info)             # Final spike detection results

# Or plot everything at once:
plot_all_debug_info(debug_info, output)
"""

'\nfrom caiman.source_extraction.volpy.spikepursuit_debug import volspike_debug\nfrom plot_debug_info import *\n\n# Run debug version on single ROI\noutput, debug_info = volspike_debug(pars, return_debug_info=True)\n\n# Choose what to plot:\nplot_intermediate_t0(debug_info, output)           # Initial trace before BG removal\nplot_t0_no_bg(debug_info, output)                  # Before/after background removal\nplot_data_hp(debug_info)                           # All pixel traces in context region\nplot_background_components(debug_info)             # SVD background components\nplot_iteration_progression(debug_info, output)     # How spikes/threshold changed per iteration\nplot_iteration_details_single(debug_info, 0)       # Detailed traces for iteration 0\nplot_final_results(output, debug_info)             # Final spike detection results\n\n# Or plot everything at once:\nplot_all_debug_info(debug_info, output)\n'

In [40]:
from caiman.source_extraction.volpy.spikepursuit_debug import volspike_debug

# Prepare input (same as volspike)
fnames = ram_path
fr = 3600
cell_n = 0
bw = ROIs[cell_n]  # 2D binary mask
weights_init = None  # or previous weights
args = {
    'template_size': 0.008,
    'context_size': 35,
    'hp_freq_pb': 1/3,
    'censor_size': 12,
    'nPC_bg': 8,
    'ridge_bg': 0.05,
    'hp_freq': 1,
    'clip': 100,
    'threshold_method': 'simple',
    'min_spikes': 10,
    'pnorm': 0.1,
    'threshold': 4,
    'sigmas': np.array([1, 1.5, 2]),
    'n_iter': 2,
    'weight_update': 'ridge',
    'do_plot': True,
    'do_cross_val': False,
    'sub_freq': 20,
    'min_width': 0,
    'max_width': 20, #10
    'w_h_ratio': 0, #2
    'visualize_ROI': True,
    'polarity': 'positive',
}
    
pars = [fnames, fr, cell_n, bw, weights_init, args]

# Get output with debug info
output, debug_info = volspike_debug(pars, return_debug_info=True)

# Inspect intermediate values
print("Initial trace stats:")
print(f"  Min: {np.min(debug_info['intermediate_t0'])}")
print(f"  Max: {np.max(debug_info['intermediate_t0'])}")

# Check iteration details
for iter_detail in debug_info['iteration_details']:
    print(f"Iteration {iter_detail['iteration']}: {iter_detail['num_spikes']} spikes, threshold={iter_detail['threshold']:.4f}")


debug_info.keys()

plot_all_debug_info(debug_info, output)


Now processing cell number 0
Parameters: template_size=0.008, context_size=35, hp_freq_pb=0.3333333333333333
Spike detection: method=simple, min_spikes=10, pnorm=0.1
Iterations: n_iter=2, weight_update=ridge
Width/height filtering: min_width=0ms, max_width=20ms, w_h_ratio=0, polarity=positive


FileNotFoundError: [Errno 2] No such file or directory: 'R:\\FOV1_T14_Green_rig__d1_18_d2_1108_d3_1_order_C_frames_36000.mmap'

In [44]:
plot_all_debug_info(debug_info, output)

Plotting all debug information...

t0 statistics:
  Shape: (36000,)
  Min: -64.6777, Max: 39.2968
  Mean: -0.0000, Std: 0.5627


Background removal statistics:
  t0 (before): std=0.5627
  t0_no_bg (after): std=0.3659
  BG component: std=0.3186


data_hp statistics:
  Shape: (36000, 810) (T=36000 frames, N_pixels=810)
  Contains: 810 different pixel traces, each with 36000 timepoints
  Min: -102.3818, Max: 109.2592
  Mean per pixel: [-5.52623278e-05 -8.62967223e-04  4.02406295e-04  9.46537039e-05
  3.07110604e-04 -3.43942898e-04 -1.01571008e-04  1.25499710e-03
  7.54268316e-04 -6.80310128e-04 -1.86504156e-04  4.32856992e-04
  8.81186512e-04 -2.07228633e-03 -4.66493075e-04  2.63889204e-04
 -1.26482453e-04  2.61122797e-04  1.10088626e-03  1.31515739e-03
  6.95333350e-04 -7.46073376e-04  9.36354976e-04  1.12491124e-03
  2.27591256e-04  1.06677040e-03 -8.76000733e-04  1.32855098e-03
  7.60227631e-05 -1.76557992e-03 -1.02241989e-03 -1.06269750e-03
  1.27260177e-03  2.25653686e-03 -1.85640547

C:\Users\strenglab\AppData\Local\Temp\ipykernel_5800\4081272041.py:387: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [42]:
#Plot data_hp from debug_info
data_hp = debug_info['data_hp']
plt.figure()
plt.plot(data_hp)
plt.xlabel('Time')
plt.ylabel('High-pass filtered signal')
plt.title('High-pass filtered signal from debug_info')
plt.show()

In [ ]:

###NEW SECTION FOR ROI COORDINATE EXTRACTION
cell_centers = [((y1 + y2) // 2, (x1 + x2) // 2) for (y1, x1, y2, x2) in Coords]
cell_centers = np.array(cell_centers)
print("Cell centers:", cell_centers)    
#display the cell centers on the image
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.imshow(img, cmap='gray') # Display the image
# ax.scatter(cell_centers[:, 1], cell_centers[:, 0], color='red') # Display the cell centers
# ax.set_title('Cell centers')    # Set the title of the plot
#plt.savefig(fname[:-4] + '_cell_centers.png', format='png', bbox_inches='tight', pad_inches=0)
 # Save the figure and close the plot     

# Save to a file
save_path = fname[:-6] + '_cell_centers.npy'
np.save(save_path, cell_centers)

print(f"Cell centers saved to {save_path}")

#check if ROIS are empty and if so skip and save error
if ROIs.shape[0] == 0:
    print("No ROIs found.")
    raise ValueError("No ROIs detected, skipping further analysis for this trial.")
else:
    print(f"Found {ROIs.shape[0]} ROIs.")


cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False, maxtasksperchild=1)

##
ROIs = ROIs                                   # region of interests
index = list(range(len(ROIs)))                # index of neurons
weights = None                                # if None, use ROIs for initialization; to reuse weights check reuse weights block

template_size = 0.008                         # half size of the window length for spike templates, default is 20 ms
context_size = 35                             # number of pixels surrounding the ROI to censor from the background PCA
visualize_ROI = False                         # whether to visualize the region of interest inside the context region
hp_freq_pb = 1/3                              # parameter for high-pass filter to remove photobleaching
clip = 100                                    # maximum number of spikes to form spike template
threshold_method = 'simple'                   # adaptive_threshold or simple
min_spikes= 10                                # minimal spikes to be found
pnorm = 0.5                                   # a variable deciding the amount of spikes chosen for adaptive threshold method
threshold = 4                                 # threshold for finding spikes only used in simple threshold method, Increase the threshold to find less spikes
do_plot = False                               # plot detail of spikes, template for the last iteration
ridge_bg= 0.05                                # ridge regression regularizer strength for background removement, larger value specifies stronger regularization
sub_freq = 20                                 # frequency for subthreshold extraction
weight_update = 'ridge'                       # ridge or NMF for weight update
n_iter = 2                                    # number of iterations alternating between estimating spike times and spatial filters
censor_size = 5                               # size of the censoring region around the ROI
min_width = 0                                 #minumum half peak-height width in ms
max_width = 20                                #maximum half peak-height width in ms      
w_h_ratio = 2                                 #minumum ratio of height in %dF/F over half peak-height width in ms
                
correl_cutoff = 0.8
snr_thresh_display = 2

opts_dict={'fnames': ram_path,   #'fnames': fname_new,
        'ROIs': ROIs,
        'fr': fr,
        'index': index,
        'weights': weights,
        'min_width': min_width,
        'max_width': max_width,
        'w_h_ratio': w_h_ratio,
        'template_size': template_size,
        'context_size': context_size,
        'visualize_ROI': visualize_ROI,
        'hp_freq_pb': hp_freq_pb,
        'clip': clip,
        'threshold_method': threshold_method,
        'min_spikes':min_spikes,
        'pnorm': pnorm,
        'threshold': threshold,
        'do_plot':do_plot,
        'ridge_bg':ridge_bg,
        'sub_freq': sub_freq,
        'weight_update': weight_update,
        'n_iter': n_iter,
        'censor_size': censor_size}

#opts.change_params(params_dict=opts_dict)
opts = volparams(params_dict=opts_dict)

vpy = VOLPY(n_processes=n_processes, dview=dview, params=opts)

print("Running VOLPY fit...")
vpy.fit(n_processes=n_processes, dview=dview)
#takes a while to run
print("Done.")


Cell centers: [[  8 477]
 [  7 540]
 [  9 563]
 [  7 764]
 [  7 418]
 [  6 362]
 [  6 496]
 [  8 289]]
Cell centers saved to D:\pAce\BKV009\20260512\FOV1_T14\FOV1_T14_Green_cell_centers.npy
Found 8 ROIs.
Running VOLPY fit...
Starting VOLPY spike detection...
Done.


In [25]:
fname[:-6]


'D:\\pAce\\BKV009\\20260512\\FOV1_T14\\FOV1_T14_Green'

In [42]:

# Visualize spatial footprints and traces
#print(np.where(vpy.estimates['locality'])[0])    # neurons that pass locality test
# idx = np.where(vpy.estimates['locality'] > 0)[0]
# utils.view_components(vpy.estimates, img_corr, idx)


##

# Reconstructed movie
# flip_signal = True    
# mv_all = utils.reconstructed_movie(vpy.estimates.copy(), fnames=mc.mmap_file,
#                                         idx=idx, scope=(0,1000), flip_signal=flip_signal)
#mv_all.play(fr=40, magnification=3)

##
vpy.estimates['ROIs'] = ROIs
vpy.estimates['Coords'] = Coords
# save_name = fname[:-4]+'_volpy'
# np.save(save_name, vpy.estimates)

cm.stop_server(dview=dview)
log_files = glob.glob('*_LOG_*')
for log_file in log_files:
    os.remove(log_file)

# print("Saved VOLPY estimates to:", save_name + '.npy')


print(vpy.estimates.keys())
print(len(vpy.estimates['spikes']))
#print(len(vpy.estimates['spikeTimes']))
print(vpy.estimates['snr']) 

#print length of each key's data:
for key in vpy.estimates.keys():
    print(f"{key}: {len(vpy.estimates[key])}")

#print number of neurons with snr > snr_thresh_display
high_snr_neurons = np.sum(vpy.estimates['snr'] > snr_thresh_display)
print(f"Number of neurons with SNR > {snr_thresh_display}: {high_snr_neurons}")



dict_keys(['rawROI', 'mean_im', 'cell_n', 't', 'ts', 't_rec', 't_sub', 'spikes', 'low_spikes', 'num_spikes', 'templates', 'snr', 'thresh', 'weights', 'locality', 'context_coord', 'F0', 'dFF', 'polarity', 'ROIs', 'Coords'])
8
[-18.28650686732223 4.822941393384902 4.955529924958627 0
 9.503352310011698 19.40608132256532 17.785471847394007 0]
rawROI: 8
mean_im: 8
cell_n: 8
t: 8
ts: 8
t_rec: 8
t_sub: 8
spikes: 8
low_spikes: 8
num_spikes: 8
templates: 8
snr: 8
thresh: 8
weights: 8
locality: 8
context_coord: 8
F0: 8
dFF: 8
polarity: 8
ROIs: 8
Coords: 8
Number of neurons with SNR > 2: 5


In [ ]:

##
vpy = vpy.estimates
#vpy['spikes'] = np.array(vpy['spikes'], dtype=object)

# try:
num_frames = np.max(vpy['dFF'].shape)
dur = num_frames/640
vpy['snr_over_thresh'] = []

vpy['raster'] = np.zeros_like(vpy['dFF'])
vpy['firing_rate'] = np.zeros_like(vpy['dFF'])
vpy['unique_trace'] = []
vpy['cell_idxs'] = []

if vpy['spikes'].size > 0:

    for i in range(vpy['dFF'].shape[0]-1):
        vpy['raster'][i, vpy['spikes'][i]] = 1
        vpy['firing_rate'][i] = savgol_filter(np.convolve(vpy['raster'][i]*640,np.ones(32)/32,mode='same'),64,1)

    for i in range(len(vpy['ROIs'])):
        vpy['snr_over_thresh'].append(abs(vpy['snr'][i]) >= snr_thresh_display) #################################################################################################################
    print("SNR LIST", vpy['snr'])
    print("snr_over_thresh", vpy['snr_over_thresh'])
    print("Number of neurons with SNR > 0:", np.sum(vpy['snr_over_thresh']))

    if np.sum(vpy['snr_over_thresh']) > 0:
        to_remove = set()
        dFF = np.array(vpy['dFF']).astype(float)
        R = np.corrcoef(dFF)
        idx0, idx1 = np.where(np.triu(R, 1) > correl_cutoff) #################################################################################################################
        max_vals = np.max(dFF, axis=1)
        smaller = np.where(max_vals[idx0] < max_vals[idx1], idx0, idx1)
        to_remove.update(smaller.tolist())
        vpy['unique_trace'] = [True if x not in to_remove else False for x in range(len(vpy['ROIs']))]

    print(vpy['unique_trace'])
    print("Correl cutoff", correl_cutoff)
    print("There are", np.sum(vpy['unique_trace']), "unique traces after correlation filtering.")
    #print("And there were ", len(to_remove), "traces removed due to high correlation.")

    vpy['cell_idxs'] = []
    for cell in range(len(vpy['ROIs'])):
        if vpy['snr_over_thresh'][cell] and vpy['unique_trace'][cell]:
            vpy['cell_idxs'].append(cell)

    print("Final number of cells after SNR and correlation filtering:", len(vpy['cell_idxs']))
    print(vpy['cell_idxs'])
    print(len(vpy['cell_idxs']))
else:
    print("No spikes > threshold in this trial.")
    raise ValueError("No spikes detected, skipping further analysis for this trial.")

wheel_mat = os.path.dirname(fname) + '\\Wheel.mat'
if os.path.exists(wheel_mat):
    wheel=mat73.loadmat(wheel_mat)
    print("Loaded wheel data from:", wheel_mat)
else:
    print("No wheel data found at:", wheel_mat)
    wheel = None




In [ ]:

#make figure
plotdata(vpy, dur, img, ROIs, fname, rootpath, unique_save_string, num_frames, mouseID, date, trialname, wheel)


In [ ]:

print("Saving VOLPY data to MAT file...")
vpy['ROIs'] = ROIs
#vpy['rect'] = r['rois']
vpy['img'] = img
del vpy['rawROI']
#scipy.io.savemat(fname[:-4] + '_volpy.mat', {'vpy': vpy}, format='5', do_compression=True)

print("Converting data types for fast saving...")

# Add 1 to cell_idxs (MATLAB 1-based indexing)
if 'cell_idxs' in vpy:
    vpy['cell_idxs'] = [x + 1 for x in vpy['cell_idxs']]

# ---------------------------------------------------------
# Helper Function for Jagged Arrays
# ---------------------------------------------------------
def make_safe_object_array(data):
    """Bypasses NumPy broadcasting to safely package irregular lists/arrays."""
    if data is None:
        return np.empty(0, dtype=object)
    
    # If it's a single item, wrap it in a list to iterate
    if not isinstance(data, (list, tuple, np.ndarray)):
        data = [data]
        
    length = len(data)
    safe_arr = np.empty(length, dtype=object)
    for i in range(length):
        # If it's a sub-list (like spike times), ensure it's converted
        if isinstance(data[i], list):
            safe_arr[i] = np.array(data[i])
        else:
            safe_arr[i] = data[i]
    return safe_arr

# ---------------------------------------------------------
# 1. Process Float Conversions
# ---------------------------------------------------------
keys_to_convert_float = [
    't', 'ts', 't_rec', 't_sub', 'templates', 'snr', 
    'thresh', 'weights', 'locality', 'context_coord', 'F0', 'dFF', 
    'raster', 'firing_rate'
]

for key in keys_to_convert_float:
    if key in vpy and vpy[key] is not None:
        try:
            vpy[key] = np.array(vpy[key], dtype=np.float32)
            print(f"  Converted '{key}' to float32 array.")
        except ValueError:
            print(f"  Could not convert '{key}' to float32. Forcing safe object array.")
            vpy[key] = make_safe_object_array(vpy[key])

# ---------------------------------------------------------
# 2. Process Integer Conversions
# ---------------------------------------------------------
keys_to_convert_int = ['num_spikes']

for key in keys_to_convert_int:
    if key in vpy and vpy[key] is not None:
        try:
            vpy[key] = np.array(vpy[key], dtype=np.int32)
            print(f"  Converted '{key}' to int32 array.")
        except ValueError:
            print(f"  Could not convert '{key}' to int32. Forcing safe object array.")
            vpy[key] = make_safe_object_array(vpy[key])

# ---------------------------------------------------------
# 3. Process Jagged Object Arrays (mean_im, spikes, etc.)
# ---------------------------------------------------------
jagged_keys = ['mean_im', 'cell_n', 'polarity', 'spikes', 'low_spikes']

for key in jagged_keys:
    if key in vpy and vpy[key] is not None:
        try:
            # Check if low_spikes happens to be a clean boolean array first
            if key == 'low_spikes' and isinstance(vpy[key], np.ndarray) and vpy[key].dtype == bool:
                continue # Leave it alone, it's fine
                
            vpy[key] = make_safe_object_array(vpy[key])
            print(f"  Safely packaged '{key}' as a NumPy object array.")
        except Exception as e:
            print(f"  Warning: Could not process '{key}': {e}")

print("Data type conversion complete.")

def clean_none(data, name="root"):
    if data is None:
        # Print the name of the key that has the None value
        print(f"Replacing None with [] at: {name}")
        return [] 
    
    elif isinstance(data, dict):
        # Recursively clean each key, passing the key name down for the print statement
        return {k: clean_none(v, name=f"{name} -> {k}") for k, v in data.items()}
    
    elif isinstance(data, list):
        # Recursively clean each list item, passing the index for the print statement
        return [clean_none(v, name=f"{name}[{i}]") for i, v in enumerate(data)]
    
    return data

vpy = clean_none(vpy)

print("Data type conversion complete.")

scipy.io.savemat(rootpath + unique_save_string + '.mat', {'vpy': vpy}, format='5', do_compression=True)
print("Saved VOLPY data to:", fname[:-4] + '_volpy.mat')


# vpy.estimates['params'] = opts
# save_name = f'volpy_{os.path.split(fnames)[1][:-5]}_{threshold_method}'
# np.save(fnames[:-4] + '_volpy.npy', vpy.estimates)

# del vpy
# # % STOP CLUSTER and clean up log files

# log_files = glob.glob('*_LOG_*')
# for log_file in log_files:
#     os.remove(log_file)
# # except ValueError as e:
# #     print(e)
# #     print("No volpy data was saved")

# # Cleanup R:/ drive
# print("Cleaning up R:/ drive...")
# def safe_close_mmap(arr):
#     try:
#         if hasattr(arr, "base") and hasattr(arr.base, "close"):
#             arr.base.close()
#     except Exception as e:
#         print("close failed:", e)

# gc.collect()  # force Python to release the memory mapping

# # 2. Delete all files in R:/
# for f in Path(r'R:/').glob('*'):
#     if f.is_file():
#         f.unlink()
# print("Cleared all files from R:/")

#Append new row to MASTERLOG
today_str = datetime.now().strftime("%Y%m%d%H%M%S")  # compact datetime string
print(today_str)
new_row = [version, today_str, unique_save_string]

with open(log_csv_path, mode='a', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(new_row)
print(f"Added new row to MasterAnalysisLOG.csv: {new_row}")



    # print(f"ERROR processing {fname}: {e}")
    # #Append new row to MASTERLOG
    # today_str = datetime.now().strftime("%Y%m%d%H%M%S")  # compact datetime string
    # new_row = [version, today_str, unique_save_string, e]

    # with open(log_csv_path, mode='a', newline='') as f:
    #     writer = csv.writer(f)
    #     writer.writerow(new_row)
    # print(f"Added new ERROR row to MasterAnalysisLOG.csv: {new_row}")
